# Exercise 3

In [ ]:
from functions import RandomnessTests
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import random
random.seed(42)
import time
import tracemalloc
np.random.seed(42)
rng = np.random.default_rng(42)

## Part 1

### Exponential Distribution

In [ ]:
def simulate_exponential(lam, n):
    U = rng.random(n)          # n uniforms
    X = -np.log(U) / lam           # inversion
    return X

exp_samples = simulate_exponential(lam=2, n=10000)

### Normal Distribution (at least with standard Box-Mueller)

In [ ]:
def box_muller_normal(n):
    U1 = rng.random(n // 2)
    U2 = rng.random(n // 2)

    R = np.sqrt(-2 * np.log(U1))
    Z1 = R * np.cos(2 * np.pi * U2)
    Z2 = R * np.sin(2 * np.pi * U2)

    Z = np.concatenate([Z1, Z2])

    if n % 2 == 1:
        U1 = rng.random()
        U2 = rng.random()
        R = np.sqrt(-2 * np.log(U1))
        extra = R * np.cos(2 * np.pi * U2)
        Z = np.append(Z, extra)

    return Z

def box_muller_general(n, mu, sigma):
    Z = box_muller_normal(n)
    return mu + sigma * Z


In [ ]:
box_muller_samples = box_muller_general(n=10000, mu=0, sigma=1)

### Pareto Distribution

In [ ]:
def simulate_pareto(n, k, beta=1.0):
    U = rng.random(n)
    X = beta * (U ** (-1.0 / k))
    return X

In [ ]:
ks = [2.05, 2.5, 3, 4]
pareto_samples = {k: simulate_pareto(10000, k) for k in ks}

### Sammenlign

In [ ]:
def theoretical_moments(dist, params):
    if dist == "exponential":
        lam = params["lam"]
        mean = 1 / lam
        var = 1 / (lam**2)
        return mean, var

    if dist == "normal":
        mu = params["mu"]
        sigma = params["sigma"]
        mean = mu
        var = sigma**2
        return mean, var

    if dist == "pareto":
        k = params["k"]
        beta = params["beta"]
        mean = beta * k / (k - 1) if k > 1 else np.nan
        var = beta**2 * k / ((k-1)**2 * (k-2)) if k > 2 else np.nan
        return mean, var


def theoretical_cdf(x, dist, params):
    x = np.asarray(x)

    if dist == "exponential":
        lam = params["lam"]
        return np.where(x < 0, 0.0, 1 - np.exp(-lam * x))

    if dist == "normal":
        mu = params["mu"]
        sigma = params["sigma"]
        return stats.norm.cdf(x, loc=mu, scale=sigma)

    if dist == "pareto":
        k = params["k"]
        beta = params["beta"]
        return np.where(x < beta, 0.0, 1 - (beta / x) ** k)


def verify_distribution(samples, dist, params):
    emp_mean = np.mean(samples)
    emp_var = np.var(samples)

    th_mean, th_var = theoretical_moments(dist, params)
    ks_stat, ks_pvalue = stats.kstest(
        samples,
        lambda x: theoretical_cdf(x, dist, params)
    )

    print("Empirical mean:", emp_mean)
    print("Theoretical mean:", th_mean)
    print()
    print("Empirical variance:", emp_var)
    print("Theoretical variance:", th_var)
    print()
    print("KS statistic:", ks_stat)
    print("KS p-value:", ks_pvalue)


def plot_distribution_axis(ax, samples, dist, params, bins=100, title=None):
    if dist == "exponential":
        lam = params["lam"]
        xs = np.linspace(0, np.max(samples), 500)
        pdf = lam * np.exp(-lam * xs)

    elif dist == "normal":
        mu = params["mu"]
        sigma = params["sigma"]
        xs = np.linspace(np.min(samples), np.max(samples), 500)
        pdf = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((xs - mu) / sigma) ** 2)

    elif dist == "pareto":
        k = params["k"]
        beta = params["beta"]
        xs = np.linspace(beta, np.max(samples), 500)
        pdf = np.where(xs >= beta, (k * beta**k) / (xs ** (k + 1)), 0.0)

    else:
        raise ValueError(f"Unknown distribution: {dist}")

    ax.hist(samples, bins=bins, density=True, alpha=0.5, label="Empirical")
    ax.plot(xs, pdf, "r", label="Theoretical PDF")
    if title is not None:
        ax.set_title(title)
    ax.legend()


def plot_distribution(samples, dist, params, bins=100):
    fig, ax = plt.subplots(figsize=(7, 4))
    plot_distribution_axis(ax, samples, dist, params, bins=bins)
    plt.tight_layout()
    plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_distribution_axis(axes[0], exp_samples, "exponential", {"lam": 2}, title="Exponential distribution, $\lambda=2$")
plot_distribution_axis(axes[1], box_muller_samples, "normal", {"mu": 0, "sigma": 1}, title="Normal distribution, $\mu=0$, $\sigma=1$")
plt.tight_layout()
plt.show()

verify_distribution(exp_samples, "exponential", {"lam": 2})
print()
verify_distribution(box_muller_samples, "normal", {"mu": 0, "sigma": 1})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, k in zip(axes, ks):
    plot_distribution_axis(ax, pareto_samples[k], "pareto", {"k": k, "beta": 1}, title=f"Pareto distribution, $beta=1$, $k={k}$")
    ax.set_xlim(0,10)
    ax.set_ylim(0, 5)

plt.tight_layout()
plt.show()

for k in ks:
    verify_distribution(pareto_samples[k], "pareto", {"k": k, "beta": 1})
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
samples_list = [exp_samples, box_muller_samples, pareto_samples[2.5]]
dists = ["exponential", "normal", "pareto"]
params_list = [{"lam": 2}, {"mu": 0, "sigma": 1}, {"k": 2.5, "beta": 1}]
titles = ["Exponential (\u03bb=2)", "Normal (0,1)", "Pareto (k=2.5, \u03b2=1)"]

for ax, samples, dist, params, title in zip(axes, samples_list, dists, params_list, titles):
    plot_distribution_axis(ax, samples, dist, params, bins=100, title=title)
    low = np.percentile(samples, 0.5)
    high = np.percentile(samples, 99.5)
    ax.set_xlim(max(low, np.min(samples)), high)

plt.tight_layout()
plt.show()

print("Goodness-of-fit (KS test):")
for dist, samples, params in zip(dists, samples_list, params_list):
    ks_stat, ks_p = stats.kstest(samples, lambda x: theoretical_cdf(x, dist, params))
    print(f"{dist}: KS stat = {ks_stat:.4f}, p-value = {ks_p:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharey=True)
axes = axes.ravel()

pareto_order = ks  # [2.05, 2.5, 3, 4]
samples_list = [exp_samples, box_muller_samples] + [pareto_samples[k] for k in pareto_order]
dists = ["exponential", "normal"] + ["pareto"] * len(pareto_order)
params_list = [{"lam": 2}, {"mu": 0, "sigma": 1}] + [{"k": k, "beta": 1} for k in pareto_order]
titles = ["Exponential (\u03bb=2)", "Normal (0,1)"] + [f"Pareto (k={k}, \u03b2=1)" for k in pareto_order]

def get_scipy_rv(dist, params):
    if dist == "exponential":
        lam = params["lam"]
        return stats.expon(scale=1/lam)
    if dist == "normal":
        return stats.norm(loc=params["mu"], scale=params["sigma"])
    if dist == "pareto":
        k = params["k"]
        beta = params.get("beta", 1)
        return stats.pareto(b=k, scale=beta)
    raise ValueError(f"Unknown dist: {dist}")

def chi2_equal_prob_bins(samples, dist, params, bins=20):
    rv = get_scipy_rv(dist, params)
    n = len(samples)
    qs = np.linspace(0, 1, bins + 1)
    edges = rv.ppf(qs)
    edges[0] = min(np.min(samples), edges[0]) if np.isfinite(edges[0]) else np.min(samples)
    edges[-1] = np.inf
    counts, _ = np.histogram(samples, bins=edges)
    expected = np.ones_like(counts) * (n / bins)
    chi_stat, p_value = stats.chisquare(f_obs=counts, f_exp=expected)
    return chi_stat, p_value, counts, expected, edges

for ax, samples, dist, params, title in zip(axes, samples_list, dists, params_list, titles):
    plot_distribution_axis(ax, samples, dist, params, bins=100, title=title)
    if dist == "pareto":
        low = params.get("beta", np.min(samples))
        high = np.percentile(samples, 99.5)
        ax.set_xlim(low, high)
    else:
        low = np.percentile(samples, 0.5)
        high = np.percentile(samples, 99.5)
        ax.set_xlim(max(low, np.min(samples)), high)

for i in range(len(samples_list), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("Goodness-of-fit (KS and Chi-squared tests) (including all Pareto k):")
for dist, samples, params in zip(dists, samples_list, params_list):
    ks_stat, ks_p = stats.kstest(samples, lambda x: theoretical_cdf(x, dist, params))
    chi_stat, chi_p, counts, expected, edges = chi2_equal_prob_bins(samples, dist, params, bins=20)
    print(f"{dist} {params}: KS stat = {ks_stat:.4f}, p-value = {ks_p:.4f}")
    print(f"{dist} {params}: Chi-squared stat = {chi_stat:.4f}, p-value = {chi_p:.4f}")

## Part 2

In [ ]:
def estimate_probability(samples, condition):
    indicators = np.array([condition(x) for x in samples])
    return np.mean(indicators)

def estimate_expectation(samples, g):
    values = np.array([g(x) for x in samples])
    return np.mean(values)

def confidence_interval(samples, alpha=0.05):
    mean = np.mean(samples)
    std = np.std(samples, ddof=1)
    n = len(samples)
    z = 1.96  # for 95%
    return mean - z*std/np.sqrt(n), mean + z*std/np.sqrt(n)

In [ ]:
X = simulate_pareto(100000, k=2.5, beta=1)

# P(X > 10)
p = estimate_probability(X, lambda x: x > 10)

# E[X]
m = estimate_expectation(X, lambda x: x)

# CI for E[X]
ci = confidence_interval(X)


In [ ]:
verify_distribution(pareto_samples[2.5], "pareto", {"k": 2.5, "beta": 1})
print("P(X > 10):", p)
print("E[X]:", m)
print("CI for E[X]:", ci)
plot_distribution(pareto_samples[2.5], "pareto", {"k": 2.5, "beta": 1})

## Part 3

In [ ]:
def ci_mean_and_var_normal(mu_true=0.0, sigma2_true=1.0, n=10, n_rep=100):
    t_975 = 2.262      # t_{0.975, 9}
    chi2_975 = 19.02   # chi^2_{0.975, 9}
    chi2_025 = 2.70    # chi^2_{0.025, 9}

    mean_intervals = []
    var_intervals = []
    mean_contains = 0
    var_contains = 0

    for _ in range(n_rep):
        x = box_muller_normal(n)

        x_bar = np.mean(x)
        s2 = np.var(x, ddof=1)

        half_width_mean = t_975 * np.sqrt(s2 / n)
        ci_mean = (x_bar - half_width_mean, x_bar + half_width_mean)

        df = n - 1
        ci_var = (df * s2 / chi2_975, df * s2 / chi2_025)

        mean_intervals.append(ci_mean)
        var_intervals.append(ci_var)

        if ci_mean[0] <= mu_true <= ci_mean[1]:
            mean_contains += 1
        if ci_var[0] <= sigma2_true <= ci_var[1]:
            var_contains += 1

    return mean_intervals, var_intervals, mean_contains, var_contains

In [ ]:
mean_ints, var_ints, mean_hits, var_hits = ci_mean_and_var_normal()

print("Mean CIs containing true mean:", mean_hits, "ud af 100")
print("Var CIs containing true variance:", var_hits, "ud af 100")


## Part 4

In [ ]:
def sample_pareto_composition(beta, k, n, rng):
    lam = rng.gamma(shape=k, scale=1.0 / beta, size=n)
    X   = rng.exponential(scale=1.0 / lam) 
    return X + beta 

In [ ]:
X1 = simulate_pareto(100000, k=4)        
X2 = sample_pareto_composition(beta=1, k=4, n=100000, rng=np.random)

print(np.mean(X1), np.mean(X2))
print(np.var(X1), np.var(X2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)
axes = axes.ravel()

k = 4

X1 = simulate_pareto(100000, k=k)        
X2 = sample_pareto_composition(beta=1, k=k, n=100000, rng=np.random)

plot_distribution_axis(axes[0], X1, "pareto", {"k": k, "beta": 1}, title=f"Pareto({k}) - Direct Inversion Method")
plot_distribution_axis(axes[1], X2, "pareto", {"k": k, "beta": 1}, title=f"Pareto({k}) - Composition Method")
axes[0].set_xlim(1, 6)
# axes[0].set_ylim(0, 5)
axes[1].set_xlim(1, 6)
# axes[1].set_ylim(0, 5)

xs = np.linspace(1, 5, 500)
pdf = k * (1 ** k) / (xs ** (k + 1))
axes[0].plot(xs, pdf, "r", label="Theoretical PDF")
axes[1].plot(xs, pdf, "r", label="Theoretical PDF")

ks_stat1, ks_p1 = stats.kstest(X1, lambda x: theoretical_cdf(x, "pareto", {"k": k, "beta": 1}))
ks_stat2, ks_p2 = stats.kstest(X2, lambda x: theoretical_cdf(x, "pareto", {"k": k, "beta": 1}))
print(f"Direct Inversion Method: KS stat = {ks_stat1:.4f}, p-value = {ks_p1:.4f}")
print(f"Composition Method: KS stat = {ks_stat2:.4f}, p-value = {ks_p2:.4f}")

def chi2_equal_prob_bins(samples, dist, params, bins=20):
    rv = get_scipy_rv(dist, params)
    n = len(samples)
    qs = np.linspace(0, 1, bins + 1)
    edges = rv.ppf(qs)
    edges[0] = min(np.min(samples), edges[0]) if np.isfinite(edges[0]) else np.min(samples)
    edges[-1] = np.inf
    counts, _ = np.histogram(samples, bins=edges)
    expected = np.ones_like(counts) * (n / bins)
    chi_stat, p_value = stats.chisquare(f_obs=counts, f_exp=expected)
    return chi_stat, p_value

chi_stat1, chi_p1 = chi2_equal_prob_bins(X1, "pareto", {"k": k, "beta": 1}, bins=20)
chi_stat2, chi_p2 = chi2_equal_prob_bins(X2, "pareto", {"k": k, "beta": 1}, bins=20)
print(f"Direct Inversion Method: Chi-squared stat = {chi_stat1:.4f}, p-value = {chi_p1:.4f}")
print(f"Composition Method: Chi-squared stat = {chi_stat2:.4f}, p-value = {chi_p2:.4f}")

t_stat, t_p = stats.ttest_ind(X1, X2, equal_var=False)
print(f"T-test for difference in means: t-statistic = {t_stat:.4f}, p-value = {t_p:.4f}")